In [ ]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from scipy import stats
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ============================================================
# 통계 검정 기반 변수 선택
# ============================================================
train = pd.read_csv('train_v8.csv')
train_raw = pd.read_csv('train.csv')
test = pd.read_csv('test_v8.csv')
submission = pd.read_csv('sample_submission.csv')

y = train['completed'].astype(int)

print("=" * 70)
print("변수별 통계 검정 (completed=0 vs completed=1)")
print("=" * 70)

group_0 = train[y == 0]
group_1 = train[y == 1]
print(group_0.shape)
print(group_1.shape)
results = []

for col in train.columns:
    if col in ['completed']:
        continue
    
    dtype = train[col].dtype
    
    if dtype in ['int64', 'float64', 'bool']:
        # 숫자형: Mann-Whitney U test (비모수)
        vals_0 = group_0[col].dropna()
        vals_1 = group_1[col].dropna()
        if len(vals_0) > 5 and len(vals_1) > 5:
            stat, pval = stats.mannwhitneyu(vals_0, vals_1, alternative='two-sided')
            test_name = 'Mann-Whitney'
        else:
            pval = 1.0
            test_name = 'skip'
    elif dtype == 'object':
        # 범주형: Chi-squared test
        ct = pd.crosstab(train[col].fillna('_NA_'), y)
        if ct.shape[0] > 1 and ct.shape[1] > 1:
            stat, pval, dof, expected = stats.chi2_contingency(ct)
            test_name = 'Chi-squared'
        else:
            pval = 1.0
            test_name = 'skip'
    else:
        continue
    
    sig = "✅" if pval < 0.05 else "❌"
    results.append({
        'feature': col,
        'test': test_name,
        'p_value': pval,
        'significant': pval < 0.05,
    })

res_df = pd.DataFrame(results).sort_values('p_value')

print(f"\n{'Feature':40s} {'Test':15s} {'p-value':>10s}  판정")
print("-" * 75)
for _, row in res_df.iterrows():
    sig = "✅ 유의" if row['significant'] else "❌ 무의미"
    print(f"  {row['feature']:38s} {row['test']:15s} {row['p_value']:10.6f}  {sig}")

# 유의한 변수만 추출
sig_features = res_df[res_df['significant']]['feature'].tolist()
nonsig_features = res_df[~res_df['significant']]['feature'].tolist()

print(f"\n 유의한 변수 ({len(sig_features)}개): {sig_features}")
print(f"무의미한 변수 ({len(nonsig_features)}개): {nonsig_features}")


In [ ]:
# ============================================================
# 완전한 검정 (v8 피처 + 원본 범주형 모두)
# ============================================================
g0_v8 = train[y == 0]
g1_v8 = train[y == 1]
g0_raw = train_raw[train_raw['completed'] == 0]
g1_raw = train_raw[train_raw['completed'] == 1]

results = []
skip_cols = ['ID', 'completed', 'generation', 'is_train']

# 1) v8의 숫자형
print("── 숫자형 (Mann-Whitney) ──")
for col in train.select_dtypes(include=['int64', 'float64', 'bool']).columns:
    if col in skip_cols: continue
    v0 = g0_v8[col].dropna()
    v1 = g1_v8[col].dropna()
    if len(v0) > 5 and len(v1) > 5:
        _, pval = stats.mannwhitneyu(v0, v1, alternative='two-sided')
    else:
        pval = 1.0
    results.append({'feature': col, 'test': 'Mann-Whitney', 'p_value': pval, 'type': '숫자'})

# 2) 원본의 범주형 (Chi-squared)
print("── 범주형 (Chi-squared) ──")
for col in train_raw.select_dtypes(include='object').columns:
    if col in skip_cols: continue
    filled = train_raw[col].fillna('_NA_')
    ct = pd.crosstab(filled, y)
    if ct.shape[0] >= 2 and ct.shape[1] >= 2:
        _, pval, _, _ = stats.chi2_contingency(ct)
        test = 'Chi-squared'
    else:
        pval, test = 1.0, 'skip'
    results.append({'feature': col, 'test': test, 'p_value': pval, 'type': '범주'})

res = pd.DataFrame(results).sort_values('p_value')

print(f"\n{'Feature':40s} {'Type':5s} {'Test':15s} {'p-value':>10s}  판정")
print("-" * 80)
for _, r in res.iterrows():
    sig = "✅" if r['p_value'] < 0.05 else ("⚠️" if r['p_value'] < 0.1 else "❌")
    print(f"  {r['feature']:38s} {r['type']:5s} {r['test']:15s} {r['p_value']:10.6f}  {sig}")

sig = res[res['p_value'] < 0.05]['feature'].tolist()
brd = res[(res['p_value'] >= 0.05) & (res['p_value'] < 0.1)]['feature'].tolist()
print(f"\n 유의 (p<0.05) {len(sig)}개: {sig}")
print(f" 경계 (p<0.1) {len(brd)}개: {brd}")


In [ ]:
# ============================================================
# p-value 기반 피처 + 모델 (v18)
# ============================================================
train = train.drop(columns=['completed'])

# ============================================================
# p<0.05 유의한 피처만 선택
# ============================================================
sig_features = [
    # 숫자형 (Mann-Whitney 유의)
    'previous_class_count', 're_registration', 'major_data',
    # 범주형 (Chi-squared 유의)  
    'certificate_acquisition', 'whyBDA', 'inflow_route',
    'major1_2',
    # 경계선 (p<0.1) - 포함
    'count_class', 'hope_for_group',
]
# previous_class_3~8은 결측 많아서 count로 대체 (이미 previous_class_count에 반영)

sig_features = [c for c in sig_features if c in train.columns]
print(f"유의 피처 {len(sig_features)}개: {sig_features}")

train_df = train[sig_features].copy()
test_df = test[sig_features].copy()

# 전처리
df = pd.concat([train_df, test_df], ignore_index=True)

for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('_missing_').astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

X = df.iloc[:len(train_df)].reset_index(drop=True)
X_test = df.iloc[len(train_df):].reset_index(drop=True)
print(f"X: {X.shape}, X_test: {X_test.shape}")

# ============================================================
# Optuna (가벼운 정규화 강한 모델)
# ============================================================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
pos_rate = float(y.mean())

def find_thr(oof):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.15, 0.65, 0.005):
        pred = (oof >= t).astype(int)
        s = f1_score(y, pred)
        if s > best_f1:
            best_f1, best_t = s, round(float(t), 3)
    return best_f1, best_t

def rf_obj(trial):
    p = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 50),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 30),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample']),
        'random_state': 42, 'n_jobs': -1,
    }
    oof = np.zeros(len(X))
    for tr, val in skf.split(X, y):
        m = RandomForestClassifier(**p)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[val] = m.predict_proba(X.iloc[val])[:, 1]
    return find_thr(oof)[0]

print("\nRF (80 trials)...")
st_rf = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
st_rf.optimize(rf_obj, n_trials=80, show_progress_bar=True)
print(f"RF: {st_rf.best_value:.4f}")

def xgb_obj(trial):
    p = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 30),
        'gamma': trial.suggest_float('gamma', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.5, 4.0),
        'random_state': 42, 'eval_metric': 'logloss', 'n_jobs': -1,
    }
    oof = np.zeros(len(X))
    for tr, val in skf.split(X, y):
        m = XGBClassifier(**p)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[val], y.iloc[val])], verbose=False)
        oof[val] = m.predict_proba(X.iloc[val])[:, 1]
    return find_thr(oof)[0]

print("\nXGB (80 trials)...")
st_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
st_xgb.optimize(xgb_obj, n_trials=80, show_progress_bar=True)
print(f"XGB: {st_xgb.best_value:.4f}")

def cat_obj(trial):
    p = {
        'iterations': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 2, 6),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 3.0, 30.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 15, 60),
        'random_strength': trial.suggest_float('random_strength', 2.0, 15.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 3.0),
        'random_seed': 42, 'verbose': 0,
        'early_stopping_rounds': 200, 'eval_metric': 'Logloss',
    }
    oof = np.zeros(len(X))
    for tr, val in skf.split(X, y):
        m = CatBoostClassifier(**p)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=(X.iloc[val], y.iloc[val]), verbose=0)
        oof[val] = m.predict_proba(X.iloc[val])[:, 1]
    return find_thr(oof)[0]

print("\nCAT (60 trials)...")
st_cat = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
st_cat.optimize(cat_obj, n_trials=60, show_progress_bar=True)
print(f"CAT: {st_cat.best_value:.4f}")

# ============================================================
# OOF + 예측
# ============================================================
from sklearn.base import clone

def get_oof(name, cls, params):
    oof = np.zeros(len(X))
    tst = np.zeros(len(X_test))
    for tr, val in skf.split(X, y):
        m = cls(**params)
        if name == 'XGB':
            m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[val], y.iloc[val])], verbose=False)
        elif name == 'CAT':
            m.fit(X.iloc[tr], y.iloc[tr], eval_set=(X.iloc[val], y.iloc[val]), verbose=0)
        else:
            m.fit(X.iloc[tr], y.iloc[tr])
        oof[val] = m.predict_proba(X.iloc[val])[:, 1]
        tst += m.predict_proba(X_test)[:, 1] / N_SPLITS
    f1, t = find_thr(oof)
    print(f"  {name} → F1: {f1:.4f}, thr: {t}")
    return oof, tst, f1, t

rf_p = {**st_rf.best_params, 'random_state': 42, 'n_jobs': -1}
xgb_p = {**st_xgb.best_params, 'random_state': 42, 'eval_metric': 'logloss', 'n_jobs': -1}
cat_p = {**st_cat.best_params, 'iterations': 2000, 'early_stopping_rounds': 200,
         'eval_metric': 'Logloss', 'random_seed': 42, 'verbose': 0}

print("\n OOF 수집")
rf_oof, rf_tst, rf_f1, rf_t = get_oof('RF', RandomForestClassifier, rf_p)
xgb_oof, xgb_tst, xgb_f1, xgb_t = get_oof('XGB', XGBClassifier, xgb_p)
cat_oof, cat_tst, cat_f1, cat_t = get_oof('CAT', CatBoostClassifier, cat_p)

# 블렌딩
blend_oof = (rf_oof + xgb_oof + cat_oof) / 3
blend_f1, blend_t = find_thr(blend_oof)
blend_tst = (rf_tst + xgb_tst + cat_tst) / 3
print(f"  Blend → F1: {blend_f1:.4f}, thr: {blend_t}")

# ============================================================
# 비율 기반 예측
# ============================================================
print("\n 비율 기반 예측")
best_oof = blend_oof if blend_f1 >= max(rf_f1, xgb_f1, cat_f1) else \
           (rf_oof if rf_f1 >= max(xgb_f1, cat_f1) else 
            (xgb_oof if xgb_f1 >= cat_f1 else cat_oof))
best_tst = blend_tst if blend_f1 >= max(rf_f1, xgb_f1, cat_f1) else \
           (rf_tst if rf_f1 >= max(xgb_f1, cat_f1) else 
            (xgb_tst if xgb_f1 >= cat_f1 else cat_tst))
best_name = 'Blend' if blend_f1 >= max(rf_f1, xgb_f1, cat_f1) else \
            ('RF' if rf_f1 >= max(xgb_f1, cat_f1) else ('XGB' if xgb_f1 >= cat_f1 else 'CAT'))

# 비율 기반: 상위 30%를 1로
target_n = int(len(best_tst) * pos_rate)
thr_pct = np.sort(best_tst)[::-1][target_n - 1]
pred_pct = (best_tst >= thr_pct).astype(int)

print(f"  {best_name} → OOF F1: {max(blend_f1, rf_f1, xgb_f1, cat_f1):.4f}")
print(f"  비율기반 thr: {thr_pct:.4f}, 1={pred_pct.sum()}, pos_rate={pred_pct.mean():.3f}")

submission['completed'] = pred_pct
submission.to_csv('submission_v18.csv', index=False)
print(f"\nsubmission_v18.csv 저장 (pos_rate={pred_pct.mean():.3f})")
